# 03 - Clean up to a consistent cell and nucleus pair

In the paper this pairing is a demonstration: on the mouse salivary gland it was used to make each nucleus fit inside one cell and to count how many cells contain one. No conclusions in the paper rest on it, and its accuracy has not been evaluated against ground truth; use it to inspect and curate, and judge the pairing on your own data.


Notebook 02 measured how well a cell segmentation and a nuclei segmentation agree. This notebook makes them agree. It is a deliberately modest clean-up: it does not try to fix every merged cell, it makes the two volumes consistent and tidy so that every nucleus sits inside exactly one cell.

We trust the **cell** labels and make the **nuclei** conform:

- Each nucleus is assigned to the cell it is mostly inside. By default (`SPLIT_STRADDLING = True`) a nucleus that straddles a cell boundary is **cut at that boundary**, so each cell keeps its own slice as a nucleus (the majority cell keeps the original id, the other cells get new ids). A slice smaller than `MIN_PIECE_FRAC` of the nucleus is thrown away, so small registration errors cannot create a sliver nucleus. Set `SPLIT_STRADDLING = False` to give the whole nucleus to its majority cell instead.
- Holes in cells are filled, and cells can optionally be smoothed.
- A cell with **no nucleus** can be rescued: the notebook looks in the nuclear stain for a dim or deep nucleus that the segmentation missed (local rescale plus threshold).
- Tiny cell fragments, and nuclei that lie outside every cell, are dropped.

Cells that still hold two or more nuclei are **kept as they are and reported**. Telling a real merge from a dividing cell needs a judgement that this notebook does not make.

The result is a consistent pair, `cleanup_cell_labels` and `cleanup_nuclei_labels`, written as `.npy` and BigTIFF files in the same format as the engine's `global_labels`. It works on any cell and nuclei pair that share one coordinate frame.

## 1. Paths and settings

Edit the two run folders. A run folder is the folder that notebook 01 prints as `Output:`, that is `WORK_DIR/OUTPUT_NAME` (not `WORK_DIR` itself). Use the same pair of runs as in notebook 02 (see the top of that notebook for how to make the nuclei run).

- `CELL_RUN_DIR`: the run of the **cell** segmentation. It must contain `stage4_full_stitch/global_labels_clean.npy`. To rescue missed nuclei (section 3) or to look at the raw image (section 5) it also needs its `tiles/` folder, and the stack that run was made from must include the nuclear stain as one of its channels.
- `NUCLEI_RUN_DIR`: the run of the **nuclei** segmentation of the same stack, with `stage4_full_stitch/global_labels_clean.npy`. The two volumes must have the same shape.

**Check the nuclear channel before you rescue.** Section 3 reads the channel `NUCLEI_CHANNEL` as the nuclear stain. Nuclei are bright blobs and the membrane is a web, so you can check by looking: run sections 1 and 2 with `RECOVER = False` (or leave `RECOVER = True`, then stop before section 3), tick the nuclei box in section 5, and only then run section 3.

In [ ]:
import sys
from pathlib import Path

# Make `packages.epicycle_scaledseg` importable when the notebook is opened from the
# repository root or from tutorials/segmentation/ (not needed if you ran `pip install -e .`).
for _p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_p / "packages" / "epicycle_scaledseg").is_dir():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break
else:
    print("Could not find the EpiCycle repository above", Path.cwd(),
          "- open the notebook from inside the repository, or run: pip install -e .")

In [ ]:
from pathlib import Path
import numpy as np

CELL_RUN_DIR   = Path(r"<EDIT>/work/cells_output")     # the CELL segmentation run
NUCLEI_RUN_DIR = Path(r"<EDIT>/work/nuclei_output")   # the NUCLEI segmentation run
VOXEL_SIZE_UM  = None        # optional (z, y, x) in micrometres

# knobs
SMOOTH       = False         # smoothing changes cell volume and shape, so keep it OFF for a mask you
                             # will measure (holes are still filled). Set True for pictures only.
SIGMA        = 1.5
MIN_CELL_VOX = 100            # drop cell fragments smaller than this (0 = keep all)
RECOVER      = True          # rescue missed nuclei in no-nucleus cells (needs the cell run's tiles/ folder)
SPLIT_STRADDLING = True      # a nucleus straddling 2+ cells is CUT at the cell boundary so each cell
                             # keeps its own slice (the majority cell keeps the original id, the
                             # others get new ids). False = the whole nucleus goes to its majority
                             # cell and the other cell is marked no_nucleus_strung_victim.
MIN_PIECE_FRAC = 0.2         # a slice must be at least this fraction of the nucleus's in-cell voxels
                             # to become its own nucleus; a smaller sliver is discarded. Only used
                             # when SPLIT_STRADDLING is True.
SMOOTH_SPLIT_NUCLEI = True   # smooth ONLY the carved nucleus pieces (both halves) so the flat cut face
                             # rounds into a nucleus-like blob; other nuclei are not touched. False =
                             # keep the flat cut. Carved ids are listed per cell in `carved_nucleus_ids`
                             # in the triage table: leave them out of any nuclear SHAPE statistic.
NUCLEI_CHANNEL = 0           # which raw channel is the NUCLEAR stain: its position in the stack, counting
                             # from 0. None = use the channel recorded in the cell run's segmentation
                             # settings. CHECK IT before section 3: look at that channel in the viewer
                             # in section 5 (nuclei are bright blobs, membrane is a web).

# --- compute resources ---
PARALLEL     = True          # use several cores for the per-cell pass (the gain is largest with SMOOTH = True)
N_WORKERS    = None          # None = all but one of the cores this process may use. Set an integer to fix it.

CELL_LABELS_NPY   = CELL_RUN_DIR   / "stage4_full_stitch" / "global_labels_clean.npy"
NUCLEI_LABELS_NPY = NUCLEI_RUN_DIR / "stage4_full_stitch" / "global_labels_clean.npy"
CELL_TILES        = CELL_RUN_DIR   / "tiles"          # raw channels (for nucleus recovery)
for p in (CELL_LABELS_NPY, NUCLEI_LABELS_NPY):
    print(p, "OK" if p.exists() else "** MISSING **")
    if not p.exists():
        folder = p.parent if p.parent.exists() else (CELL_RUN_DIR if p == CELL_LABELS_NPY else NUCLEI_RUN_DIR)
        print("   that folder contains:", sorted(q.name for q in folder.iterdir()) if folder.exists() else "(the folder does not exist)")

In [ ]:
if VOXEL_SIZE_UM is None:
    print("WARNING: VOXEL_SIZE_UM is None.")
    print("  Every feature written in section 6 (areas, volumes, lengths) will be in VOXEL units, not")
    print("  micrometres. Voxels are rarely cubic, so shape features will be wrong for any analysis")
    print("  that assumes physical units. Set VOXEL_SIZE_UM above (z, y, x) and re-run.")
else:
    print(f"Voxel size (z, y, x) in um: {VOXEL_SIZE_UM}")

## 2. Carve and smooth to a consistent pair (no rescue yet)

In [ ]:
from packages.epicycle_scaledseg.reconcile import cleanup_volume

assert "<EDIT>" not in str(CELL_RUN_DIR) and "<EDIT>" not in str(NUCLEI_RUN_DIR), "edit the run folders in section 1"
print("loading the cell and nuclei label volumes (silent, may take a while for large volumes)...", flush=True)
cells  = np.load(str(CELL_LABELS_NPY))      # full copy in RAM
nuclei = np.load(str(NUCLEI_LABELS_NPY))
assert cells.shape == nuclei.shape, f"cell {cells.shape} != nuclei {nuclei.shape} - not co-registered"
print(f"loaded. cells {cells.shape} {cells.dtype} (max id {int(cells.max()):,}) ; "
      f"nuclei max id {int(nuclei.max()):,}", flush=True)   # max() not unique() (no full sort)

# Carve, assign each nucleus to its majority cell, and (optionally) smooth. progress=True prints a
# progress bar for each phase. The *_path arguments let the worker processes read the .npy files
# directly. Rescue of missed nuclei is the separate pass in section 3.
# return_triage=True also returns `triage`: one status row per surviving cell (used in section 6).
# With SMOOTH = False the per-cell work is small, so you will see the cores only lightly used.
out_cells, out_nuclei, report, triage = cleanup_volume(
    cells, nuclei, nuclei_raw=None, smooth=SMOOTH, sigma=SIGMA,
    min_cell_vox=MIN_CELL_VOX, recover=False,
    split_straddling_nuclei=SPLIT_STRADDLING, min_piece_frac=MIN_PIECE_FRAC,
    smooth_split_nuclei=SMOOTH_SPLIT_NUCLEI,
    parallel=PARALLEL, n_workers=N_WORKERS,
    cells_path=str(CELL_LABELS_NPY), nuclei_path=str(NUCLEI_LABELS_NPY),
    progress=True, return_triage=True)

print("\n=== cleanup report ===")
for k, v in sorted(report.items()):
    print(f"  {k:22s} {v:,}")
print("verifying consistency (full-volume scan - silent)...", flush=True)
assert (out_nuclei[out_cells == 0] == 0).all(), "a nucleus sticks out of all cells!"
print("consistency OK: every nucleus is inside a cell.")
print(f"Left as they are: {report.get('merged_cell', 0):,} cells still hold 2 or more nuclei (possible merges or dividing cells).")

## 3. Rescue missed nuclei in cells that have none  *(optional)*

For each cell with no nucleus, the notebook reads the **nuclear stain** inside the cell's bounding box from the tiles (without loading the whole volume) and tries to pull out a dim or deep nucleus by a local contrast rescale and a threshold. Rescued nuclei get new ids. This needs the cell run's `tiles/` folder with the nuclear stain among its channels, and `NUCLEI_CHANNEL` set correctly (see section 1); if it is missing, the cell below skips the rescue and says so.

Rescue can add false positives (debris, signal bleeding in from a neighbour), so look at a sample before trusting it.

In [ ]:
if RECOVER and not CELL_TILES.exists():
    print(f"no tiles folder at {CELL_TILES}: skipping the rescue.")
    RECOVER = False
if RECOVER:
    from scipy.ndimage import find_objects
    from packages.epicycle_scaledseg.reconcile import (read_global_roi, recover_nucleus, load_geometry,
                                    resolve_channel_layout)
    try:
        from tqdm.auto import tqdm
    except Exception:
        def tqdm(it, **k): return it
    geo = load_geometry(CELL_TILES); n_ch = geo['channels']
    lay = resolve_channel_layout(CELL_TILES)        # read the channel layout the run was made with
    auto_nuc = lay.get('nucleus_channel')           # the nucleus channel recorded in the run
    # an explicit NUCLEI_CHANNEL wins; otherwise use the nucleus channel recorded in the run
    if NUCLEI_CHANNEL is not None:
        nuc_ch = int(NUCLEI_CHANNEL)
    elif auto_nuc is not None:
        nuc_ch = int(auto_nuc)
    else:
        nuc_ch = None
    if n_ch < 2:
        print(f"cell run has {n_ch} channel(s) -> no nuclear channel; SKIPPING recovery.")
        RECOVER = False
    elif nuc_ch is None:
        print(f"no nuclei channel found in the run settings (layout={lay}). Set NUCLEI_CHANNEL in "
              f"section 1 to the index of the nuclear stain (the channel's position in the stack, counting from 0). "
              f"Skipping the rescue for now.")
        RECOVER = False
    elif not (0 <= nuc_ch < n_ch):
        raise ValueError(f"NUCLEI_CHANNEL={nuc_ch} out of range for a {n_ch}-channel run")
    else:
        src = "NUCLEI_CHANNEL override" if NUCLEI_CHANNEL is not None else "the run's segmentation settings"
        assert out_cells.shape == (geo['depth'], geo['full'][0], geo['full'][1]), \
            "label volume != tile frame (not co-registered)"
        print(f"reading nuclear channel = {nuc_ch} (from {src}; signal/membrane = {lay.get('signal_channel')}). "
              f"Look at channel {nuc_ch} once to confirm it shows bright nuclei blobs.")
        print("indexing cells for recovery (full-volume scan, silent)...", flush=True)
        has_nuc = np.zeros(int(out_cells.max()) + 1, dtype=bool)
        has_nuc[np.unique(out_cells[out_nuclei > 0])] = True
        Z, H, W = out_cells.shape; nxt = int(out_nuclei.max()) + 1; n_rec = 0
        tri_by_cell = {r["cell_id"]: r for r in triage}   # patch triage rows as nuclei are recovered
        # only the NO-NUCLEUS cells need a (slow) per-bbox stain read -> bar over just those
        targets = [(idx + 1, sl) for idx, sl in enumerate(find_objects(out_cells))
                   if sl is not None and not has_nuc[idx + 1]]
        print(f"{len(targets):,} no-nucleus cells to attempt recovery on (one at a time, each reads from disk)")
        for cid, sl in tqdm(targets, desc="recovering nuclei"):
            z0, z1 = max(0, sl[0].start - 1), min(Z, sl[0].stop + 1)
            y0, y1 = max(0, sl[1].start - 2), min(H, sl[1].stop + 2)
            x0, x1 = max(0, sl[2].start - 2), min(W, sl[2].stop + 2)
            cmask = out_cells[z0:z1, y0:y1, x0:x1] == cid
            rec = recover_nucleus(read_global_roi(CELL_TILES, (z0, z1, y0, y1, x0, x1), nuc_ch), cmask)
            if rec is not None:
                out_nuclei[z0:z1, y0:y1, x0:x1][rec & cmask] = nxt
                r = tri_by_cell.get(cid)              # no_nucleus_* -> recovered_nucleus (keep triage true)
                if r is not None:
                    r["status"] = "recovered_nucleus"; r["n_nuclei"] = 1; r["nucleus_ids"] = [nxt]
                nxt += 1; n_rec += 1
        assert (out_nuclei[out_cells == 0] == 0).all(), "recovery wrote a nucleus outside a cell!"
        print(f"recovered {n_rec:,} missed nuclei. CAVEAT: recovery can inject false positives "
              f"(debris/bleed-in) - look at a sample before trusting. {len(targets) - n_rec:,} "
              f"cells still have no nucleus (kept + flagged).")
else:
    print("RECOVER = False: skipping the rescue.")

In [ ]:
# --- Final tally: how many cells end up WITH a nucleus (after the clean-up and the rescue) ---
from collections import Counter as _Counter
if "triage" not in globals():
    print("no `triage` in memory: run sections 2 and 3 first.")
else:
    _sc = _Counter(r["status"] for r in triage)
    _tot = len(triage)
    _HAS = ("clean", "split_nucleus", "recovered_nucleus", "merged_cell")   # >=1 nucleus
    _NONE = ("no_nucleus_strung_victim", "no_nucleus_genuine")               # 0 nuclei
    _has = sum(_sc.get(k, 0) for k in _HAS)
    _none = sum(_sc.get(k, 0) for k in _NONE)
    print("=== FINAL TALLY (after cleanup + recovery) ===")
    print(f"  {_tot:,} surviving cells")
    print(f"  {_has:,} have a nucleus ({100 * _has / max(_tot, 1):.1f}%)   |   "
          f"{_none:,} have none ({100 * _none / max(_tot, 1):.1f}%)")
    print("  breakdown by status:")
    for _k in _HAS + _NONE:
        print(f"     {_k:26s} {_sc.get(_k, 0):>10,}")
    print(f"  residual to curate (merged_cell = 2+ nuclei): {_sc.get('merged_cell', 0):,}")
    if "report" in globals():
        print(f"  (also dropped earlier in section 2: {report.get('dropped_fragment', 0):,} fragment cells, "
              f"{report.get('orphan_nucleus_dropped', 0):,} orphan nuclei with no cell)")

## 4. Write the consensus  *(set WRITE = True)*

Check the report above first. Then set `WRITE = True` and run the cell. Files go to `CELL_RUN_DIR/reconcile/`: `cleanup_cell_labels` and `cleanup_nuclei_labels`, each as `.npy` and as a BigTIFF.

In [ ]:
WRITE = False        # <-- set True to save the cleaned, consistent pair
WRITE_TIFF = True    # also write the BigTIFF copies (set False to save disk space; the .npy files are what the other cells read)
WROTE_NPY = False
if WRITE:
    import shutil
    import tifffile
    out_dir = CELL_RUN_DIR / "reconcile"; out_dir.mkdir(exist_ok=True)
    n_files = 4 if WRITE_TIFF else 2
    need = n_files / 2 * (out_cells.size + out_nuclei.size) * 4          # int32 files
    free = shutil.disk_usage(out_dir).free
    print(f"about to write {n_files} files, {need / 1e9:,.1f} GB in total; free space in {out_dir}: {free / 1e9:,.1f} GB")
    assert free > need * 1.05, "not enough free disk space for the consensus volumes (set WRITE_TIFF = False, or free some space)"
    for name, vol in [("cell", out_cells), ("nuclei", out_nuclei)]:
        for ext in ("npy", "tif"):
            if (out_dir / f"cleanup_{name}_labels.{ext}").exists():
                print(f"note: cleanup_{name}_labels.{ext} already exists and will be overwritten")
        arr = np.ascontiguousarray(vol, dtype=np.int32)
        print(f"writing cleanup_{name}_labels (silent, can take a few minutes)...", flush=True)
        np.save(out_dir / f"cleanup_{name}_labels.npy", arr)
        if WRITE_TIFF:
            tifffile.imwrite(str(out_dir / f"cleanup_{name}_labels.tif"), arr, bigtiff=True)
        print(f"  wrote cleanup_{name}_labels", flush=True)
    WROTE_NPY = True
    print(f"DONE. cleanup_cell_labels + cleanup_nuclei_labels in {out_dir}")
    print("every nucleus is inside exactly one cell; cells with 2 or more nuclei are left as they are.")
else:
    print("WRITE = False: nothing written. Review the report, then set it to True.")

## 5. Look at the consensus  *(optional)*

A quick look inside the notebook, with matplotlib and ipywidgets. Pick a view (XY, XZ or YZ), scrub through the slices, and zoom into a region. Tick the raw-channel boxes to overlay the membrane and nuclear stain (these read from the tiles, so zoom in first if a full slice is slow). Cell labels are drawn as outlines and nuclei as solid colour; the opacity slider changes the label colour strength.

This needs `ipywidgets` (`pip install ipywidgets`). The viewer shows the result in memory (sections 2 and 3) if there is one, and otherwise the files saved by section 4; it prints which one it is showing.

In [ ]:
try:
    import ipywidgets  # noqa: F401
except ImportError:
    ipywidgets = None

_recon = CELL_RUN_DIR / "reconcile"
_have_mem = "out_cells" in globals() and "out_nuclei" in globals()
_have_disk = (_recon / "cleanup_cell_labels.npy").exists() and (_recon / "cleanup_nuclei_labels.npy").exists()
if ipywidgets is None:
    print('ipywidgets is not installed (pip install ipywidgets): skipping the viewer.')
elif not (_have_mem or _have_disk):
    print("Nothing to show yet: run section 2 first (or write the consensus in section 4).")
else:
    # --- Inline slice viewer (matplotlib + ipywidgets, drawn in this notebook) ---
    # Labels are memory-mapped; each raw slice is read on demand from the tiles (only the viewed crop).
    try:
        get_ipython().run_line_magic("matplotlib", "inline")   # ensure inline backend (no-op outside IPython)
    except Exception:
        pass
    import matplotlib.pyplot as plt
    from ipywidgets import (interactive_output, HBox, VBox, Dropdown, IntSlider, FloatSlider,
                            SelectionSlider, Checkbox)
    from IPython.display import display
    from skimage.segmentation import find_boundaries

    MAX_DISPLAY_PX = 2000   # cap the rendered image's largest side (raise it to render higher-res)

    # 1) labels: the result in memory if there is one, otherwise the saved files (memory-mapped)
    recon_dir = _recon
    if _have_mem:
        CELLS, NUC = out_cells, out_nuclei
        print("labels: showing the result in memory (sections 2 and 3)")
    else:
        CELLS = np.load(str(recon_dir / "cleanup_cell_labels.npy"), mmap_mode="r")
        NUC = np.load(str(recon_dir / "cleanup_nuclei_labels.npy"), mmap_mode="r")
        print(f"labels: nothing in memory, showing the saved files in {recon_dir}")
    ZZ, YY, XX = CELLS.shape
    print(f"volume dims (Z, Y, X) = {(ZZ, YY, XX)}")

    # per-view display metadata: which volume axis is rows(vertical) / cols(horizontal) / the slice, and
    # the physical pixel aspect (rows-spacing / cols-spacing) so ortho (XZ/YZ) aren't squished. Set
    # VOXEL_SIZE_UM=(z,y,x) in section 1 for physically-correct proportions; else voxels are treated as cubes.
    _SP = tuple(float(v) for v in VOXEL_SIZE_UM) if VOXEL_SIZE_UM else (1.0, 1.0, 1.0)   # (z, y, x)
    _AX = {"XY": ("Y", "X", "Z"), "XZ": ("Z", "X", "Y"), "YZ": ("Z", "Y", "X")}          # (rows, cols, slice)
    _ASPECT = {"XY": _SP[1] / _SP[2], "XZ": _SP[0] / _SP[2], "YZ": _SP[0] / _SP[1]}       # rows/cols spacing

    # 2) raw channels - lazy per-slice read off the tiles; best-effort (labels still work without it)
    RAW = {}
    try:
        from packages.epicycle_scaledseg.reconcile import load_geometry, resolve_channel_layout, read_global_roi
        _geo = load_geometry(CELL_TILES); _lay = resolve_channel_layout(CELL_TILES)
        _nch = int(_geo["channels"])
        _frame = (int(_geo["depth"]), int(_geo["full"][0]), int(_geo["full"][1]))
        if _frame != (ZZ, YY, XX):        # co-registration guard: raw ROIs are built from label dims
            raise ValueError(f"label shape {(ZZ, YY, XX)} != tile frame {_frame} (not co-registered)")
        _nuc = int(NUCLEI_CHANNEL) if NUCLEI_CHANNEL is not None else _lay.get("nucleus_channel")
        _sig = _lay.get("signal_channel")
        if _sig is None and _nuc is not None and _nch == 2:
            _sig = 1 - int(_nuc)
        _cand = {"membrane": _sig, "nuclei_raw": _nuc}
        RAW = {k: int(v) for k, v in _cand.items() if v is not None and 0 <= int(v) < _nch}
        _skip = {k: v for k, v in _cand.items() if k not in RAW}
        print(f"raw channels available: {RAW}" + (f"  (skipped invalid {_skip})" if _skip else "")
              + "  (membrane -> green, nuclei -> purple)")
    except Exception as e:
        print(f"raw channels unavailable ({type(e).__name__}: {e}) -> labels only")

    # 3) stable random colours for the label overlays
    _LUT_C = np.random.default_rng(0).random((50000, 3))
    _LUT_N = np.random.default_rng(1).random((50000, 3))
    _chan_cache = {}

    def _norm(a, lo_p=1.0, hi_p=99.7):                  # gentle white point -> less over-saturation
        a = np.asarray(a, np.float32)
        lo, hi = np.percentile(a, lo_p), np.percentile(a, hi_p)
        return np.clip((a - lo) / (hi - lo + 1e-6), 0, 1) if hi > lo else np.zeros_like(a)

    def _bounds(view, rowf, colf):
        (r0, r1), (c0, c1) = rowf, colf
        R = YY if view == "XY" else ZZ                 # rows axis length
        C = YY if view == "YZ" else XX                 # cols axis length
        a0 = min(int(r0 * R), R - 1); a1 = min(max(int(r1 * R), a0 + 1), R)   # clamp start in-bounds
        b0 = min(int(c0 * C), C - 1); b1 = min(max(int(c1 * C), b0 + 1), C)
        s = max(1, -(-max(a1 - a0, b1 - b0) // MAX_DISPLAY_PX))               # CEIL stride: cap actually holds
        return a0, a1, b0, b1, s

    def _lab_slice(vol, view, idx, b):
        a0, a1, b0, b1, s = b
        if view == "XY": return np.asarray(vol[min(idx, ZZ - 1), a0:a1:s, b0:b1:s])
        if view == "XZ": return np.asarray(vol[a0:a1:s, min(idx, YY - 1), b0:b1:s])
        return np.asarray(vol[a0:a1:s, b0:b1:s, min(idx, XX - 1)])           # YZ -> (Z, Y)

    def _raw_slice(ch, view, idx, b):
        a0, a1, b0, b1, s = b
        if view == "XY":   roi, ax = (min(idx, ZZ - 1), min(idx, ZZ - 1) + 1, a0, a1, b0, b1), 0
        elif view == "XZ": roi, ax = (a0, a1, min(idx, YY - 1), min(idx, YY - 1) + 1, b0, b1), 1
        else:              roi, ax = (a0, a1, b0, b1, min(idx, XX - 1), min(idx, XX - 1) + 1), 2
        crop = np.squeeze(np.asarray(read_global_roi(CELL_TILES, tuple(int(v) for v in roi), ch)), axis=ax)
        return crop[::s, ::s]

    def _channels(view, idx, rowf, colf, membrane, nuclei_raw, b):
        # returns (green_src, purple_src) normalized [0,1] 2D arrays or None; cached WITHOUT brightness
        key = (view, idx, tuple(rowf), tuple(colf), bool(membrane), bool(nuclei_raw))
        if key in _chan_cache:                          # reuse across brightness / label / alpha changes
            return _chan_cache[key]
        green = _norm(_raw_slice(RAW["membrane"], view, idx, b)) if (membrane and "membrane" in RAW) else None
        mag   = _norm(_raw_slice(RAW["nuclei_raw"], view, idx, b)) if (nuclei_raw and "nuclei_raw" in RAW) else None
        _chan_cache.clear(); _chan_cache[key] = (green, mag)
        return green, mag

    def _render(view, idx, rowf, colf, membrane, nuclei_raw, cells, nuclei_lab, cell_outline, brightness, alpha):
        a0, a1, b0, b1, s = _bounds(view, rowf, colf); b = (a0, a1, b0, b1, s)
        hh, ww = len(range(a0, a1, s)), len(range(b0, b1, s))               # shape without touching CELLS
        img = np.zeros((hh, ww, 3), np.float32)
        green, mag = _channels(view, idx, rowf, colf, membrane, nuclei_raw, b)
        if green is not None:
            img[..., 1] = np.clip(green * brightness, 0, 1)                  # membrane -> GREEN
        if mag is not None:
            mm = np.clip(mag * brightness, 0, 1)
            img[..., 0] = np.maximum(img[..., 0], mm); img[..., 2] = np.maximum(img[..., 2], mm)  # -> PURPLE
        if cells:                                                           # cells: outline OR solid
            cl = _lab_slice(CELLS, view, idx, b)
            if cell_outline:
                mask = find_boundaries(cl, mode="inner")
                if not mask.any() and (cl > 0).any():
                    mask = cl > 0                     # crop is one whole cell -> draw solid (not invisible)
            else:
                mask = cl > 0
            img[mask] = (1 - alpha) * img[mask] + alpha * _LUT_C[cl[mask] % len(_LUT_C)]
        if nuclei_lab:                                                      # nuclei: solid
            nl = _lab_slice(NUC, view, idx, b); nm = nl > 0
            img[nm] = (1 - alpha) * img[nm] + alpha * _LUT_N[nl[nm] % len(_LUT_N)]
        return np.clip(img, 0, 1)

    def _show(view, idx, zoom, cx, cy, membrane, nuclei_raw, brightness, cells, cell_outline, nuclei_lab, alpha):
        w = 1.0 / zoom                                                      # window fraction; pan to (cx,cy)
        r0 = min(max(cy - w / 2, 0.0), 1.0 - w); c0 = min(max(cx - w / 2, 0.0), 1.0 - w)
        perp = {"XY": ZZ, "XZ": YY, "YZ": XX}[view]; rax, cax, sax = _AX[view]
        img = _render(view, idx, (r0, r0 + w), (c0, c0 + w), membrane, nuclei_raw,
                      cells, nuclei_lab, cell_outline, brightness, alpha)
        fig, ax = plt.subplots(figsize=(9, 9)); ax.imshow(img, aspect=_ASPECT[view]); ax.axis("off")
        ax.set_title(f"{view}   {sax}-slice {min(idx, perp - 1)}/{perp - 1}   |   "
                     f"pan y = {rax} (vertical), pan x = {cax} (horizontal)   |   zoom {zoom}x")
        plt.show(); plt.close(fig)

    # 4) widgets - grouped: navigate | zoom/pan | channels | labels. `slice` spans the current view's
    #    perpendicular axis (resized on view change); zoom magnifies, cx/cy pan (only matter when zoomed).
    _PERP = {"XY": ZZ, "XZ": YY, "YZ": XX}
    w_view = Dropdown(options=["XY", "XZ", "YZ"], value="XY", description="view")
    w_idx  = IntSlider(ZZ // 2, 0, ZZ - 1, 1, description="slice", continuous_update=False)
    w_zoom = SelectionSlider(options=[1, 2, 4, 8, 16, 32], value=1, description="zoom")
    w_cx   = FloatSlider(0.5, min=0, max=1, step=0.02, description="pan x", continuous_update=False)
    w_cy   = FloatSlider(0.5, min=0, max=1, step=0.02, description="pan y", continuous_update=False)
    w_mem  = Checkbox(value=False, description="membrane (green)")
    w_nraw = Checkbox(value=False, description="nuclei (purple)")
    w_bri  = FloatSlider(1.0, min=0.2, max=4.0, step=0.1, description="brightness", continuous_update=False)
    w_cell = Checkbox(value=True, description="cell labels")
    w_out  = Checkbox(value=True, description="cell outlines")
    w_nlab = Checkbox(value=True, description="nuclei labels")
    w_alpha = FloatSlider(0.6, min=0, max=1, step=0.05, description="label opacity", continuous_update=False)

    def _on_view(change):
        new_max = _PERP[change["new"]] - 1
        w_idx.max = max(w_idx.max, new_max)   # grow first so value is never transiently > max
        w_idx.value = new_max // 2            # jump to the middle plane of the new axis
        w_idx.max = new_max                   # shrink to the real per-view range
    w_view.observe(_on_view, names="value")

    _out_area = interactive_output(_show, dict(
        view=w_view, idx=w_idx, zoom=w_zoom, cx=w_cx, cy=w_cy, membrane=w_mem, nuclei_raw=w_nraw,
        brightness=w_bri, cells=w_cell, cell_outline=w_out, nuclei_lab=w_nlab, alpha=w_alpha))
    display(VBox([HBox([w_view, w_idx]),
                 HBox([w_zoom, w_cx, w_cy]),
                 HBox([w_mem, w_nraw, w_bri]),
                 HBox([w_cell, w_out, w_nlab, w_alpha])]), _out_area)
    print("On XY: pan y = Y, pan x = X. On XZ/YZ the VERTICAL axis is Z (the title spells it out per view).")
    print("Membrane=green, nuclei=purple; brightness fixes over-saturation. Cells outline (untick for solid); nuclei solid.")
    print("Ortho (XZ/YZ) proportions use VOXEL_SIZE_UM if set in section 1 (else voxels treated as cubes).")

## 6. Feature and triage tables  *(one row per cell, one per nucleus)*

This measures the shape of every cleaned **cell** and **nucleus** with the engine's morphology code, then joins each cell's clean-up `status` (`clean`, `merged_cell`, `split_nucleus`, `recovered_nucleus`, `no_nucleus_strung_victim`, `no_nucleus_genuine`) and a few nucleus summaries (`n_nuclei`, `nucleus_voxels`, `nucleus_cell_frac`). A `split_nucleus` cell is one that was rescued by a carve: its only nucleus is a freshly carved slice.

The nucleus table measures the same catalogue features on each nucleus, plus the pairing columns from the cleanup (owning cell, status, nucleus count). Paired cell-nucleus and nuclear-shape measurements are not part of this release and are not reported in the paper.

**Leaving carved nuclei out of shape statistics: use `carved_nucleus_ids`, not `status`.** Both halves of a split nucleus are carved (and, with `SMOOTH_SPLIT_NUCLEI`, smoothed at the cut), so their shape is an estimate and not a measurement. The majority half keeps `status = "clean"`, so filtering on `status` would let it through. The list `carved_nucleus_ids` names both halves; drop those nucleus ids before you quantify nuclear shape.

**What `status` can and cannot tell you.** It is a worklist of structural problems, not a complete bad-cell detector. It comes from the nucleus count after clean-up, so it reliably flags cells with two nuclei, cells with no nucleus, and orphan nuclei. By construction it misses:

- **Merges whose two nuclei were also fused into one label.** The cell reads one nucleus, so it reads `clean`.
- **Merges whose second nucleus was assigned to a neighbour.** The merged cell reads `clean` and the neighbour is marked `no_nucleus_strung_victim`, which points at the wrong object.

Shape measures such as `area`, `sphericity` and `nucleus_cell_frac` catch grotesque outliers, but a compact two-cell merge can look normal. `nucleus_cell_frac` is a soft hint, not a gate: a low value can mean a lost-nucleus merge or just a large cell. So `status = "clean"` means "one nucleus after clean-up", not "good cell".
The cell features use the EpiCycle catalogue (`feature_set="epicycle"`), which also finds which cells touch each other (the contact graph written in section 8). If you saved the consensus in section 4, the measuring reads those `.npy` files directly; otherwise each volume is first copied to a temporary file the size of the volume, so make sure your temporary folder has room.

In [ ]:
from packages.epicycle_scaledseg.reconcile import extract_label_features, assemble_cell_table, assemble_nucleus_table

# Per-label shape measures on the cleaned volumes, then join the clean-up `status` per cell.
# If section 4 wrote the .npy files in this session, measure from those (no temporary copy).
_dir = CELL_RUN_DIR / "reconcile"
if globals().get("WROTE_NPY", False):
    cells_in, nuclei_in = str(_dir / "cleanup_cell_labels.npy"), str(_dir / "cleanup_nuclei_labels.npy")
else:
    cells_in, nuclei_in = out_cells, out_nuclei
print("extracting CELL features...", flush=True)
cell_feats = extract_label_features(cells_in, spacing=VOXEL_SIZE_UM, n_workers=N_WORKERS,
                                    progress=True, desc="cell feats", feature_set="epicycle")
print("extracting NUCLEUS features...", flush=True)
nuc_feats  = extract_label_features(nuclei_in, spacing=VOXEL_SIZE_UM, n_workers=N_WORKERS,
                                    progress=True, desc="nuc feats", feature_set="epicycle")

cell_table = assemble_cell_table(triage, cell_feats, nuc_feats)   # 1 row per cell + status + nucleus summaries
nuc_table  = assemble_nucleus_table(nuc_feats, triage)            # 1 row per nucleus + owning cell_id

print(f"\ncell_table: {len(cell_table):,} cells x {cell_table.shape[1]} cols ; "
      f"nuc_table: {len(nuc_table):,} nuclei")
print("\nstatus counts (your triage worklist):")
print(cell_table['status'].value_counts().to_string())
cols = [c for c in ['label', 'status', 'n_nuclei', 'area', 'nucleus_voxels',
                    'nucleus_cell_frac', 'sphericity', 'aspect_ratio'] if c in cell_table.columns]
cell_table[cols].head(10)

## 7. Write the feature tables  *(set WRITE_FEATURES = True)*

The triage tables, one row per cell and one per nucleus. Run this cell before section 8.

In [ ]:
WRITE_FEATURES = False   # <-- set True to save the feature + triage tables
if WRITE_FEATURES:
    out_dir = CELL_RUN_DIR / "reconcile"; out_dir.mkdir(exist_ok=True)
    for name, df in [("cell", cell_table), ("nucleus", nuc_table)]:
        df.to_csv(out_dir / f"cleanup_{name}_features.csv", index=False)
        try:
            df.to_parquet(out_dir / f"cleanup_{name}_features.parquet", index=False)
        except Exception as e:
            print(f"  (parquet skipped for {name}: {e}; CSV written)")
    print(f"WROTE cleanup_cell_features + cleanup_nucleus_features (.csv + .parquet) to {out_dir}")
    print("To triage: filter `status` in {merged_cell, no_nucleus_strung_victim, "
          "no_nucleus_genuine} for the residual cells worth curating.")
else:
    print("WRITE_FEATURES = False: the tables stay in memory (cell_table, nuc_table). Set it to True to save them.")

## 8. Make the cell table usable for the analysis notebooks

The notebooks in `tutorials/analysis/` start from a per-cell feature table with a `stack_name` column, plus the contact graph of the cells (the adjacency CSV, `<stack>_adjacency.csv`). This cell writes both next to the other outputs, in `CELL_RUN_DIR/reconcile/`. The feature table is `all_stacks_features.csv`: the cell table of section 6, including the clean-up `status`. Filter on `status` (and leave out the cells listed in `carved_nucleus_ids`) before an analysis if you want only well-formed cells. `STACK_NAME` is the name the analysis notebooks will show; change it if you like.

Units are micrometres only if you set `VOXEL_SIZE_UM`; otherwise voxels.

In [ ]:
STACK_NAME = "consensus"        # EDIT: the name for this stack in the analysis notebooks

if WRITE_FEATURES:
    from packages.epicycle_scaledseg.core.qc_features import write_edges_csv
    out_dir = CELL_RUN_DIR / "reconcile"; out_dir.mkdir(exist_ok=True)
    table = cell_table.copy()
    table.insert(0, "stack_name", STACK_NAME)
    table.to_csv(out_dir / "all_stacks_features.csv", index=False)
    print(f"wrote all_stacks_features.csv  ({len(table):,} cells)")
    print("Units :", "um" if VOXEL_SIZE_UM is not None else "VOXELS (VOXEL_SIZE_UM was None)")
    edges_tmp = out_dir / "edges.csv"
    if write_edges_csv(cell_feats, edges_tmp, node_ids=cell_feats["label"].to_numpy()):
        edges_tmp.replace(out_dir / f"{STACK_NAME}_adjacency.csv")
        print(f"wrote {STACK_NAME}_adjacency.csv")
    else:
        print("no contact graph was produced (no touching cells found)")
else:
    print("WRITE_FEATURES = False: nothing written (set it to True in section 7 and run this cell again).")